# Building-to-service travel-time matrix

This notebook computes the matrix of travel times from the example buildings to the example services on an
intermodal graph (walk + public transport) with IduEdu's `od_matrix` and stores it in
`examples_data/matrix_time.parquet` for the [provision example](provision.ipynb).

In [1]:
# !pip install "iduedu[pbf]" pyarrow

In [2]:
import geopandas as gpd
import pandas as pd
from iduedu import config, download_osm_pbf, get_intermodal_graph, od_matrix
from shapely.ops import unary_union

## 1. Buildings and services

In [3]:
# Read building and service datasets
buildings = gpd.read_parquet('examples_data/buildings.parquet')
services = gpd.read_parquet('examples_data/services.parquet')

## 2. Territory

The convex hull of all buildings and services, buffered by 0.001° (about 60–110 m), defines the graph extent.

In [4]:
polygon = unary_union(
    buildings.to_crs(4326).geometry.to_list() + services.to_crs(4326).geometry.to_list()
).convex_hull.buffer(0.001)

## 3. Intermodal graph

The territory is cut out of the Geofabrik extract of the Northwestern Federal District (IduEdu >= 2.4); the
same call without the `with` block downloads the data from the Overpass API.

In [5]:
pbf = download_osm_pbf(territory=polygon, region="northwestern-fed-district")
with config.use_pbf(pbf):
    G_intermodal = get_intermodal_graph(territory=polygon, clip_by_territory=True)

2026-09-15 18:14:27.333 | WARNING  | HTTP 504: retrying in 1.0s (attempt 1/10)


2026-09-15 18:15:18.080 | WARNING  | Removing 391 nodes outside the largest strongly connected component. Retaining 21329 of 21720 nodes.


2026-09-15 18:15:18.101 | WARNING  | Public transport nodes dropped with the smaller components: {'trolleybus': 2}


## 4. Travel-time matrix

`od_matrix` snaps origins and destinations to graph nodes and runs Dijkstra searches with the edge weight `weight`
(`"time_min"`, minutes) up to `threshold` (45 minutes). Pairs beyond the threshold are unreachable.

In [6]:
# Compute travel time matrix (in minutes)
matrix: pd.DataFrame = od_matrix(
    G_intermodal,
    gdf_origins=buildings,
    gdf_destinations=services,
    weight="time_min",
    threshold=45,
)


## 5. Storage

The matrix is stored for the provision example.

In [7]:
# Save matrix to Parquet format
# IduEdu returns a sparse DataFrame to keep large OD matrices compact in memory.
# Parquet writers do not support pandas SparseDtype columns, so convert to a dense
# DataFrame before saving. Unreachable pairs stay encoded as np.inf.
matrix_dense = matrix.sparse.to_dense()
matrix_dense.to_parquet('examples_data/matrix_time.parquet')
